# 3단계: 속성(Aspect) 추출 — BERT fine-tuning (도메인별 독립 모델)

2단계 검증 후 결정한 대로, **도메인별로 별도 모델**을 학습한다 (가전/IT기기/화장품/생활/패션 각각 속성 체계가 달라서). 이 노트북은 `DOMAIN` 변수 하나만 바꾸면 5개 도메인 아무거나 학습할 수 있게 만들었다.

## Colab(GPU)에서 실행

임베딩(2단계)보다 훨씬 무거운 작업(학습=순전파+역전파 반복)이라 로컬 CPU로는 비현실적이다. 로컬에서는 50건짜리 초소형 샘플로 코드가 실제로 도는지(loss가 떨어지는지)만 드라이런으로 확인했다.

**실행 전 체크리스트**
1. 런타임 > 런타임 유형 변경 → GPU
2. 왼쪽 파일 탭에 아래 파일 업로드 (한 도메인만 학습할 거면 그 도메인 것만 올려도 됨)
   - `data/processed/aspect_datasets/{도메인}_reviews.parquet`
   - `data/processed/aspect_datasets/{도메인}_labels.json`
3. 아래 `DOMAIN` 변수를 원하는 도메인으로 설정 (`IT기기`, `가전`, `화장품`, `생활`, `패션` 중 하나)
4. 끝까지 실행 → 마지막 셀에서 결과 자동 다운로드

In [ ]:
!pip install -q "transformers>=4.40" "accelerate>=1.1.0" scikit-learn pyarrow

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
from sklearn.metrics import f1_score, classification_report

# ===== 본인이 학습할 도메인으로 변경 =====
DOMAIN = "IT기기"   # "IT기기" | "가전" | "화장품" | "생활" | "패션"
MODEL_NAME = "klue/bert-base"
DATA_DIR = "."      # 업로드한 파일이 있는 경로
OUT_DIR = f"./{DOMAIN}_model_output"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "GPU가 안 잡혔다. 런타임 > 런타임 유형 변경에서 GPU를 선택했는지 확인할 것"

## 1. 데이터 로드

리뷰별 속성 멀티핫(multi-hot) 라벨은 로컬에서 미리 만들어뒀다 (`data/processed/aspect_datasets/` 생성 과정은 `docs/personal/05_3단계_속성추출_상세노트.md` 참고).

**`split` 컬럼 주의**: 이 컬럼은 원본 AIHub 분할이 아니라 `notebooks/03a_resplit_조.ipynb`로 재분할한 결과다 — 원본 분할을 그대로 쓰면 라벨별로 검증셋에 0건인 경우가 있어서(예: 화장품 44개 중 9개), 라벨마다 검증 데이터가 최소한은 들어가도록 미리 재분할해뒀다. 이 노트북은 그 결과를 그대로 읽어서 쓸 뿐이라 재분할 코드 자체는 여기 없다 (재현하려면 `03a_resplit_조.ipynb`를 먼저 실행할 것).

In [ ]:
df = pd.read_parquet(f"{DATA_DIR}/{DOMAIN}_reviews.parquet")
with open(f"{DATA_DIR}/{DOMAIN}_labels.json", encoding="utf-8") as f:
    labels = json.load(f)

train_df = df[df["split"] == "Training"].reset_index(drop=True)
val_df = df[df["split"] == "Validation"].reset_index(drop=True)

print(f"도메인={DOMAIN}, 라벨 수={len(labels)}")
print(f"Train={len(train_df)}, Val={len(val_df)}")
print("라벨 목록:", labels)

# 라벨별 등장 빈도(=클래스 불균형 확인용, 1단계 EDA에서 이미 알고 있던 문제)
label_freq = np.stack(train_df["label_vector"].values).sum(axis=0)
for name, cnt in sorted(zip(labels, label_freq), key=lambda x: -x[1])[:5]:
    print(f"  {name}: {int(cnt)}건 ({cnt/len(train_df)*100:.1f}%)")

## 2. 모델/데이터셋 준비

`klue/bert-base` + multi-label 분류 헤드(sigmoid, 리뷰 개요에 원래 명시된 구조). 로컬 드라이런에서 이미 이 조합의 코드가 정상 동작하는 걸 확인했다.

**클래스 불균형 보정 추가됨**: 화장품 도메인 학습 결과 확인 후 추가한 부분. 흔한 라벨(가격, 기능/효과 등) 위주로 학습되고 희귀 라벨은 정밀도만 높고 재현율이 바닥을 치는 문제가 확인되어(예: "두피보호" 학습데이터 290건 있어도 F1=0), 라벨별 `pos_weight`를 준 커스텀 loss(`WeightedTrainer`)로 바꿨다 (상한 20→5 튜닝 과정은 `results/reports/03_속성추출_결과.md` 참고). **화장품·생활·패션은 이 보정을 적용해서 (재)학습됨. IT기기·가전은 미적용 상태로 남아있음** — 두 도메인 다 완전실패 라벨이 없거나(가전) 근본적 데이터 부족(IT기기 소비전력, 64건뿐이라 가중치로도 해결 안 됨)이라 우선순위를 낮췄기 때문.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class AspectDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame["RawText_clean"].tolist()
        self.label_vecs = np.stack(frame["label_vector"].values).astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True, max_length=128, padding="max_length")
        item = {k: torch.tensor(v) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.label_vecs[idx])
        return item


train_ds = AspectDataset(train_df)
val_ds = AspectDataset(val_df)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    problem_type="multi_label_classification",
)

# ===== 클래스 불균형 대응: 라벨별 pos_weight =====
# v1(상한 20)으로 화장품 학습 -> 완전 실패했던 라벨 5개는 살아났지만,
# 색상(원래 F1 0.83이던 라벨)까지 정밀도가 0.87->0.60으로 무너지는 과교정이 발생함.
# 원인: 색상처럼 데이터가 아주 적지 않은 라벨도 자연계산 pos_weight가 11배가 넘어서(20 상한에 안 걸렸는데도)
# 이미 과했음 -> 상한을 20 -> 5로 낮춰서 극희귀 라벨은 여전히 밀어주되 전반적 과교정을 줄인다.
n_train = len(train_df)
pos_counts = np.stack(train_df["label_vector"].values).sum(axis=0)
pos_weight = (n_train - pos_counts) / np.clip(pos_counts, 1, None)
pos_weight = np.clip(pos_weight, a_min=None, a_max=5.0)
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32)
print(f"pos_weight 범위: {pos_weight.min():.2f} ~ {pos_weight.max():.2f} (평균 {pos_weight.mean():.2f})")


class WeightedTrainer(Trainer):
    """기본 Trainer는 라벨 가중치 없는 BCEWithLogitsLoss를 씀 -> pos_weight 적용한 버전으로 교체."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        target = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor.to(logits.device))
        loss = loss_fct(logits, target)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)
    return {
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

## 3. 학습

epoch=3, batch=32, lr=2e-5는 BERT fine-tuning의 통상적인 시작값이다. 도메인당 3~4만 건 규모라 T4에서 각 epoch당 대략 10~20분 정도로 예상 (2단계 임베딩 속도 대비 학습이라 몇 배 더 걸림 — `docs/personal/05_3단계_속성추출_상세노트.md`에 추정 근거 정리).

In [ ]:
args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    logging_steps=50,
    report_to=[],
    fp16=True,   # GPU에서 반정밀도로 속도 향상
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

## 4. 결과 검증

전체 micro/macro F1뿐 아니라 **라벨별** 성능을 봐야 한다 — 1단계 EDA에서 이미 확인했듯 라벨 빈도가 매우 불균형해서(예: "가격"은 수만 건, 드문 속성은 몇십~몇백 건), 전체 평균 지표만 보면 소수 라벨이 아예 학습이 안 됐는데도 숫자가 괜찮아 보일 수 있다.

In [ ]:
pred_output = trainer.predict(val_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)

print("=== 전체 지표 ===")
print(pred_output.metrics)

print("\n=== 라벨별 지표 (support = 검증셋에서 실제 등장 횟수) ===")
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

# 아예 한 번도 예측되지 않은 라벨이 있는지 확인 (모델 붕괴 신호)
never_predicted = [labels[i] for i in range(len(labels)) if y_pred[:, i].sum() == 0]
print(f"\n검증셋에서 단 한 번도 예측 안 된 라벨: {never_predicted if never_predicted else '없음'}")

print("\n=== 샘플 예측 육안 확인 (5건) ===")
rng = np.random.default_rng(0)
for i in rng.choice(len(val_df), size=5, replace=False):
    true_labels = [labels[j] for j in range(len(labels)) if y_true[i][j] == 1]
    pred_labels = [labels[j] for j in range(len(labels)) if y_pred[i][j] == 1]
    print(f"\n리뷰: {val_df.iloc[i]['RawText_clean'][:60]}")
    print(f"  실제: {true_labels}")
    print(f"  예측: {pred_labels}")

## 5. 저장

모델 가중치(`klue/bert-base` 파인튜닝본, 약 440MB)는 팀 가이드대로 각자 이름 붙여서 구글드라이브에 올린다. 여기서는 로컬로 필요한 것만 다운로드한다: 라벨 목록, 평가지표, 예측 결과. 모델 전체 가중치는 용량이 커서 구글드라이브에 직접 업로드하거나, Colab 내에서 드라이브에 바로 저장하는 걸 권장 (파일 탭에서 다운받기엔 너무 큼).

In [ ]:
import os

# 모델 가중치 저장 (용량이 커서 구글드라이브 업로드 권장 -> 본인 이름 붙여서, 예: aspect_model_IT기기_조.pt)
trainer.save_model(f"{OUT_DIR}/final")
tokenizer.save_pretrained(f"{OUT_DIR}/final")

# 가벼운 결과물만 로컬로 다운로드
report_dict = classification_report(y_true, y_pred, target_names=labels, zero_division=0, output_dict=True)
with open(f"{DOMAIN}_metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "domain": DOMAIN,
        "overall": pred_output.metrics,
        "per_label": report_dict,
        "never_predicted_labels": never_predicted,
    }, f, ensure_ascii=False, indent=2)

print("저장 완료:", f"{DOMAIN}_metrics.json")

from google.colab import files
files.download(f"{DOMAIN}_metrics.json")

print("\n모델 가중치 위치:", f"{OUT_DIR}/final")
print("-> 구글드라이브에 'aspect_model_{도메인}_본인이름' 형식으로 업로드할 것 (팀 가이드 참고)")